# Student dropout data: exploratory analysis
Run after placing a CSV in `data/raw/` or uploading one through the app. Set `CSV_PATH` and `TARGET` below. This notebook reports observed data only; it does not fit a model or treat correlations as causal evidence.


In [ ]:
from pathlib import Path
import sys
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))
from profiling import profile_dataframe, detect_target
CSV_PATH = None  # e.g. ROOT / 'data/raw/uci_portuguese.csv'
TARGET = None  # e.g. 'Target'; auto-detected when None
if CSV_PATH is None:
    candidates = sorted((ROOT / 'data/raw').glob('*.csv'))
    if not candidates:
        raise FileNotFoundError('Place a CSV in data/raw or set CSV_PATH to an uploaded CSV path.')
    CSV_PATH = candidates[0]
df = pd.read_csv(CSV_PATH, low_memory=False)
TARGET = TARGET or detect_target(df)
profile = profile_dataframe(df, TARGET)
print(CSV_PATH, df.shape, 'target:', TARGET)
display(df.head())


In [ ]:
# Data quality: every feature, missingness, duplicate rows, IQR outliers
quality = pd.DataFrame([{k: c.get(k) for k in ['name','type','dtype','missing','missing_percent','unique','outlier_count']} for c in profile['columns']])
display(quality.sort_values('missing', ascending=False))
print('Exact duplicate rows:', profile['duplicate_rows'])
print('Quality issues:', *profile['quality_issues'], sep='\n- ')
quality.set_index('name')['missing'].sort_values().plot.barh(figsize=(9, max(4, len(quality)*.25)), title='Missing cells by feature')
plt.tight_layout(); plt.show()


In [ ]:
# Class distribution and feature distributions; edit IMPORTANT_FEATURES after reading the data dictionary.
if TARGET:
    display(pd.DataFrame(profile['class_distribution']))
    df[TARGET].astype('string').fillna('(missing)').value_counts().plot.bar(title='Outcome distribution')
    plt.ylabel('Students'); plt.tight_layout(); plt.show()
numeric = [c['name'] for c in profile['columns'] if c['type'] == 'numeric']
IMPORTANT_FEATURES = numeric
for name in IMPORTANT_FEATURES:
    fig, axes = plt.subplots(1, 2, figsize=(10, 3))
    sns.histplot(df[name].dropna(), ax=axes[0], bins=25)
    sns.boxplot(x=df[name], ax=axes[1])
    fig.suptitle(name); plt.tight_layout(); plt.show()


In [ ]:
# Important categorical distributions and observed dropout proportions by category.
IMPORTANT_CATEGORICAL = [c for c in ['Gender', 'Scholarship', 'Department', 'Parental_Education', 'Course', 'Debtor', 'Scholarship holder'] if c in df.columns]
for name in IMPORTANT_CATEGORICAL:
    counts = df[name].astype('string').fillna('(missing)').value_counts().head(12)
    fig, axes = plt.subplots(1, 2, figsize=(11, 3))
    counts.plot.bar(ax=axes[0], title=name + ' counts')
    if TARGET:
        dropout = df[TARGET].astype('string').str.lower().isin(['1', 'dropout'])
        rates = dropout.groupby(df[name].astype('string').fillna('(missing)')).mean().reindex(counts.index)
        rates.plot.bar(ax=axes[1], title='Observed dropout fraction', ylim=(0, 1))
    fig.suptitle(name); plt.tight_layout(); plt.show()


In [ ]:
# Correlations are descriptive; coded categories and identifiers are excluded.
if len(numeric) >= 2:
    selected = df[numeric[:20]].corr(numeric_only=True)
    plt.figure(figsize=(max(8,len(selected)*.6), max(6,len(selected)*.5)))
    sns.heatmap(selected, cmap='BrBG', center=0)
    plt.title('Pearson correlation among numeric columns'); plt.tight_layout(); plt.show()
display(pd.DataFrame(profile['correlations']).head(20))


In [ ]:
# Save reproducible profile and feature-availability manifest for model development.
import json
from feature_groups import feature_groups
source = 'kaggle_synthetic' if 'Student_ID' in df.columns else 'uci_portuguese' if 'Target' in df.columns else 'unspecified'
manifest = feature_groups(df, source)
display({name: len(cols) for name, cols in manifest['stages'].items()})
output = ROOT / 'data/processed' / f'{Path(CSV_PATH).stem}_profile.json'
output.parent.mkdir(parents=True, exist_ok=True)
output.write_text(json.dumps({'profile': profile, 'feature_groups': manifest}, indent=2), encoding='utf-8')
print('Saved', output)


## Findings to fill after running on real data
- Dataset source, version, date, license, and target meaning:
- Rows, features, class proportions, missingness, duplicates, outliers:
- Plausible data errors or impossible values confirmed by the data dictionary:
- Features available at prediction time; exclude later semester or final-outcome fields for enrollment-stage predictions:
- Differences between UCI and Kaggle datasets; avoid interpreting synthetic patterns as real-world evidence:
